<a href="https://colab.research.google.com/github/rpaivadias69/MALE_Abgaben_PVA-s_Nachbearbeitungen/blob/PVA2_RaulPaivaDias_Nachbearbeitung2/MALE_PVA2_Nachbereitungsaufgaben_RaulPaivaDias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
#CHAPTER 3:
#Exercise 1:
#Preparation / imports
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

#Load MNIST and split
mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X_all = mnist.data
y_all = mnist.target.astype(np.uint8)

X_train, X_test = X_all[:60000], X_all[60000:]
y_train, y_test = y_all[:60000], y_all[60000:]

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

#Hyperparameter
param_grid = {
    # First Try: "n_neighbors": [3, 5, 7],  <-- Gave me 96.9% accuracy
    "n_neighbors": [3, 4, 5, 6, 7], #<-- 2nd try, 97.14% accuracy
    "weights": ["uniform", "distance"]
}

knn_model = KNeighborsClassifier()

grid = GridSearchCV(
    estimator=knn_model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# Faster Search, only part of the trainingset
X_search = X_train[:12000]
y_search = y_train[:12000]

grid.fit(X_search, y_search)

print("Best Parameter:", grid.best_params_)
print("Best CV-Accuracy:", grid.best_score_)

#Train best modell on full trainingset
best_knn = KNeighborsClassifier(**grid.best_params_)
best_knn.fit(X_train, y_train)

#Evaluate Testset
y_pred = best_knn.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print(f"Test-Accuracy: {test_acc:.4f}")


(60000, 784) (10000, 784)
(60000,) (10000,)
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameter: {'n_neighbors': 4, 'weights': 'distance'}
Best CV-Accuracy: 0.9461666666666666
Test-Accuracy: 0.9714


In [23]:
#Exercise 2:
#Preparation / imports
from scipy.ndimage import shift

#Shift function
def shift_mnist_image(image, direction):
    image_2d = image.reshape(28, 28)

    if direction == "left":
        shifted = shift(image_2d, [0, -1], cval=0)
    elif direction == "right":
        shifted = shift(image_2d, [0, 1], cval=0)
    elif direction == "up":
        shifted = shift(image_2d, [-1, 0], cval=0)
    elif direction == "down":
        shifted = shift(image_2d, [1, 0], cval=0)
    else:
        raise ValueError("direction must be 'left', 'right', 'up' or 'down' ")

    return shifted.reshape(-1)

#Expand the trainingsset
directions = ["left", "right", "up", "down"]

X_train_expanded = [img for img in X_train]
y_train_expanded = [label for label in y_train]

for img, label in zip(X_train, y_train):
    for direction in directions:
        X_train_expanded.append(shift_mnist_image(img, direction))
        y_train_expanded.append(label)

X_train_expanded = np.array(X_train_expanded)
y_train_expanded = np.array(y_train_expanded)

print("Old form X_train:", X_train.shape)
print("New form X_train_expanded:", X_train_expanded.shape)
print("New form y_train_expanded:", y_train_expanded.shape)

#Retrain the modell
augmented_knn = KNeighborsClassifier(**grid.best_params_)
augmented_knn.fit(X_train_expanded, y_train_expanded)

#Evaluate the testset
y_test_pred_aug = augmented_knn.predict(X_test)
augmented_acc = accuracy_score(y_test, y_test_pred_aug)

print(f"Accuracy with expanded trainigsset: {augmented_acc:.4f}")

#Accuracy was 97.14%, now its higher

Old form X_train: (60000, 784)
New form X_train_expanded: (300000, 784)
New form y_train_expanded: (300000,)
Accuracy with expanded trainigsset: 0.9763


In [24]:
#Exercise 3
#Preparation / imports
import pandas as pd

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

#Using google colab notebook, this code below allows me to select the csv files I need and upload them !manually!
#After running, please look below in the result-cell you will find the button to "upload files"
from google.colab import files
uploaded = files.upload()

#load csv files
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

#separate target and features
X_train_titanic = train_data.drop(["Survived", "Name", "Ticket", "Cabin"], axis=1)
y_train_titanic = train_data["Survived"]

X_test_titanic = test_data.drop(["Name", "Ticket", "Cabin"], axis=1)

#define numerical and categorical columns
num_features = ["Age", "SibSp", "Parch", "Fare"]
cat_features = ["Pclass", "Sex", "Embarked"]

#preprocessing for numerical columns
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

#preprocessing for categorical columns
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

#combine preprocessing
preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

#full model pipeline
titanic_model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42
    ))
])

#evaluate with cross-validation
cv_scores = cross_val_score(
    titanic_model,
    X_train_titanic,
    y_train_titanic,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:", cv_scores)
print("Mean accuracy:", cv_scores.mean())

#fit model on full training data
titanic_model.fit(X_train_titanic, y_train_titanic)

#predict for test set
test_predictions = titanic_model.predict(X_test_titanic)

print(test_predictions[:10])

Saving test.csv to test (2).csv
Saving train.csv to train (2).csv
Cross-validation scores: [0.81005587 0.82022472 0.84269663 0.79775281 0.84831461]
Mean accuracy: 0.8238089259933463
[0 0 0 0 0 0 1 0 1 0]


In [2]:
#Exercise 4
#Same as above, google colab, therefore manually upload the folders with the mail
from google.colab import files
uploaded = files.upload()

#Since more than 500 Mails, uploading the zipped files and manually unzipping them:
import tarfile

for filename in uploaded.keys():
    if filename.endswith(".tar.bz2"):
        with tarfile.open(filename, "r:bz2") as tar:
            tar.extractall()

#Load the mails
import os

def load_emails_from_folder(folder):
    emails = []
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        with open(path, "r", errors="ignore") as f:
            emails.append(f.read())
    return emails

ham_emails = load_emails_from_folder("easy_ham")
spam_emails = load_emails_from_folder("spam")

print("Ham emails:", len(ham_emails))
print("Spam emails:", len(spam_emails))


#Train and test split
import numpy as np
from sklearn.model_selection import train_test_split

X = np.array(ham_emails + spam_emails)
y = np.array([0] * len(ham_emails) + [1] * len(spam_emails))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Extract features
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Feature matrix shape:", X_train_vec.shape)

#training classifiers

#Option 1: Naive Bayes (All commented out)
#from sklearn.naive_bayes import MultinomialNB
#from sklearn.metrics import accuracy_score, precision_score, recall_score

#nb_model = MultinomialNB()
#nb_model.fit(X_train_vec, y_train)

#y_pred_nb = nb_model.predict(X_test_vec)

#print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
#print("Precision:", precision_score(y_test, y_pred_nb))
#print("Recall:", recall_score(y_test, y_pred_nb))
#Results:
#Ham emails: 2551
#Spam emails: 501
#Feature matrix shape: (2441, 64398)
#Naive Bayes Accuracy: 0.9852700490998363
#Precision: 1.0
#Recall: 0.9021739130434783

#Option 2: Logistic Regression:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_vec, y_train)

y_pred_log = log_model.predict(X_test_vec)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))

#Results:
#Ham emails: 2551
#Spam emails: 501
#Feature matrix shape: (2441, 64398)
#Logistic Regression Accuracy: 0.9967266775777414
#Precision: 0.9891304347826086
#Recall: 0.9891304347826086


Saving 20021010_easy_ham.tar.bz2 to 20021010_easy_ham.tar (4).bz2
Saving 20021010_spam.tar.bz2 to 20021010_spam.tar (4).bz2
Ham emails: 2551
Spam emails: 501
Feature matrix shape: (2441, 64398)
Logistic Regression Accuracy: 0.9967266775777414
Precision: 0.9891304347826086
Recall: 0.9891304347826086


Chapter 4 (Questions 1-11):

Q1: Which linear regression training algorithm can you use if you have a training set with millions of features?

A1: If the dataset contains millions of features, using the Normal Equation becomes computationally expensive because it requires inverting a very large matrix.

In this case, iterative optimization methods such as Gradient Descent are more suitable.

In particular, Stochastic Gradient Descent (SGD) is a good choice because it can handle large datasets efficiently and does not require loading the entire dataset into memory at once.

Q2: Suppose the features in your training set have very different scales. Which algorithms might suffer from this, and how? What can you do about it?

A2: If features have very different scales, Gradient Descent algorithms can suffer because the cost function becomes stretched, which leads to slow convergence and inefficient updates.

To fix this, feature scaling (e.g., standardization or normalization) should be applied before training.

Q3: Can gradient descent get stuck in a local minimum when training a logistic regression model?

A3: No, Gradient Descent cannot get stuck in a local minimum when training a logistic regression model because the cost function is convex. This means there is only one global minimum.

Q4: Do all gradient descent algorithms lead to the same model, provided you let them run long enough?

A4: Not necessarily. Different Gradient Descent algorithms may converge to slightly different solutions depending on factors such as the learning rate and randomness (especially for SGD). However, they should all get close to the optimal solution if properly configured.

Q5: Suppose you use batch gradient descent and you plot the validation error at every epoch. If you notice that the validation error consistently goes up, what is likely going on? How can you fix this?

A5: If the validation error consistently increases, it is likely that the model is overfitting the training data.

This can be fixed by using regularization, reducing model complexity, or applying early stopping.

Q6: Is it a good idea to stop mini-batch gradient descent immediately when the validation error goes up?

A6: No, it is not a good idea to stop immediately. Validation error can fluctuate, especially with mini-batch or stochastic gradient descent.

Instead, early stopping with some patience should be used, meaning training stops only if the validation error does not improve for several iterations.

Q7: Which gradient descent algorithm (among those we discussed) will reach the vicinity of the optimal solution the fastest? Which will actually converge? How can you make the others converge as well?

A7: Stochastic Gradient Descent typically reaches the vicinity of the optimal solution the fastest because it updates parameters more frequently.
However, Batch Gradient Descent is more likely to fully converge to the exact minimum.

To help other methods converge, you can reduce the learning rate over time (learning rate scheduling).

Q8: Suppose you are using polynomial regression. You plot the learning curves and you notice that there is a large gap between the training error and the validation error. What is happening? What are three ways to solve this?

A8: A large gap between training error and validation error indicates overfitting.
This can be improved by:
- reducing the model complexity (e.g., using a lower polynomial degree)
- increasing regularization
- adding more training data

Q9: Suppose you are using ridge regression and you notice that the training error and the validation error are almost equal and fairly high. Would you say that the model suffers from high bias or high variance? Should you increase the regularization hyperparameter α or reduce it?

A9: If both the training error and validation error are high and similar, the model is suffering from high bias (underfitting).
In this case, the regularization parameter α should be reduced, because strong regularization makes the model too simple.

Q10: Why would you want to use:

a: Ridge regression instead of plain linear regression (i.e., without any regularization)?

b: Lasso instead of ridge regression?

c: Elastic net instead of lasso regression?

A10:

a) Ridge regression is preferred over plain linear regression because it helps reduce overfitting by penalizing large weights, which improves generalization.

b) Lasso regression can be preferred over ridge regression because it can set some feature weights exactly to zero, effectively performing feature selection.

c) Elastic Net is preferred over lasso when there are multiple correlated features, since it combines both L1 and L2 regularization and provides more stable results.

Q11: Suppose you want to classify pictures as outdoor/indoor and daytime/nighttime. Should you implement two logistic regression classifiers or one softmax regression classifier?

A11: In this case, it is better to use two logistic regression classifiers instead of one softmax classifier.
This is because the problem is multi-label (an image can be both outdoor and daytime at the same time), whereas softmax regression is designed for mutually exclusive classes.



In [3]:
#Chapter 4, Exercise 12:
#Preparation / imports
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#Load data
iris = load_iris()
X = iris.data
y = iris.target

#Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Scale feature
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

#Add bias term
X_train = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_val = np.c_[np.ones((X_val.shape[0], 1)), X_val]

#One-hot encoding
def to_one_hot(y):
    n_classes = np.max(y) + 1
    one_hot = np.zeros((len(y), n_classes))
    one_hot[np.arange(len(y)), y] = 1
    return one_hot

Y_train = to_one_hot(y_train)
Y_val = to_one_hot(y_val)

#Softmax function
def softmax(logits):
    exp_vals = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_vals / np.sum(exp_vals, axis=1, keepdims=True)

#Initialize parameters
n_features = X_train.shape[1]
n_classes = Y_train.shape[1]

theta = np.random.randn(n_features, n_classes) * 0.01

#Training settings
learning_rate = 0.1
n_epochs = 1000
best_loss = np.inf
patience = 20
patience_counter = 0

#Training loop
for epoch in range(n_epochs):

    logits = X_train @ theta
    probs = softmax(logits)

    loss = -np.mean(np.sum(Y_train * np.log(probs + 1e-9), axis=1))

    gradients = (1 / len(X_train)) * X_train.T @ (probs - Y_train)
    theta -= learning_rate * gradients

    #Validation loss
    val_logits = X_val @ theta
    val_probs = softmax(val_logits)
    val_loss = -np.mean(np.sum(Y_val * np.log(val_probs + 1e-9), axis=1))

    #Early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

#Evaluate
predictions = np.argmax(X_val @ theta, axis=1)
accuracy = np.mean(predictions == y_val)

print("Validation accuracy:", accuracy)

Validation accuracy: 1.0
